# YOLOE-26m on VisDrone: E2 full fine-tune (overnight)
Run with **Save Version → Save & Run All**, Internet ON, GPU T4×2. No inputs need to be attached.
Outputs (in this version's output): `yoloe26m_e2_full.pt`, `e2_results.csv`, `runs_yoloe/e2_full/`.

In [1]:
# --- Setup ---------------------------------------------------------------
!pip install -q "ultralytics==8.4.174" "sahi==0.12.8" pycocotools "git+https://github.com/ultralytics/CLIP.git"
import ultralytics, sahi, torch
print(ultralytics.__version__, sahi.__version__, torch.__version__, torch.cuda.device_count(), "GPU(s)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
8.4.174 0.12.8 2.11.0+cu128 2 GPU(s)


In [2]:
# --- Raw VisDrone val + test-dev (for the quick scorer) --------------------
from pathlib import Path
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import download
VD = Path("/tmp/VisDrone_raw")
download([f"{ASSETS_URL}/VisDrone2019-DET-{s}.zip" for s in ("val", "test-dev")], dir=VD, threads=2)
print(*sorted(VD.glob("*/*")), sep="\n")

Unzipping /tmp/VisDrone_raw/VisDrone2019-DET-val.zip to /tmp/VisDrone_raw/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 2.1Kfiles/s 0.5s
Unzipping /tmp/VisDrone_raw/VisDrone2019-DET-test-dev.zip to /tmp/VisDrone_raw/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.6Kfiles/s 2.1s
/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/annotations
/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/images
/tmp/VisDrone_raw/VisDrone2019-DET-val/annotations
/tmp/VisDrone_raw/VisDrone2019-DET-val/images


In [3]:
# --- Build K1's training data (D2 = ignore-masked full images + 640/960 tiles) ---
# Copied from the K1 training notebook (visdrone-training, Cells 2-3) with only the test-challenge
# download and the stats printout removed. Tiling is seeded per image, so this reproduces K1's data exactly:
# expect "D1: 6471 masked images | D2 tiles: 63745". Takes ~5 min, ~15 GB in /tmp.
import os, json, zipfile, urllib.request, cv2, random, shutil, numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

DATA = Path("/tmp/VisDrone"); DATA.mkdir(parents=True, exist_ok=True)
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
SPLITS = {"train": "VisDrone2019-DET-train", "val": "VisDrone2019-DET-val", "test-dev": "VisDrone2019-DET-test-dev"}
NAMES = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]

def download_split(name):
    z = DATA / f"{name}.zip"
    if (DATA / name).exists():
        return True
    print("Downloading", name)
    urllib.request.urlretrieve(BASE + z.name, z)
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)
    z.unlink()
    return True

def convert(split_dir):
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    lbl_dir.mkdir(exist_ok=True)
    ignores = {}
    for ann in tqdm(sorted(ann_dir.glob("*.txt")), desc=split_dir.name):
        W, H = Image.open(img_dir / f"{ann.stem}.jpg").size
        out, ign = [], []
        for row in ann.read_text().strip().splitlines():
            v = row.strip().rstrip(",").split(",")
            if len(v) < 6:
                continue
            x, y, w, h, score, cat = map(int, v[:6])
            if w <= 0 or h <= 0:
                continue
            if score == 0 or cat == 0:
                ign.append([x, y, w, h])
                continue
            if cat == 11:
                continue
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            bw, bh = (x2 - x1) / W, (y2 - y1) / H
            out.append(f"{cat-1} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (lbl_dir / ann.name).write_text("\n".join(out))
        ignores[ann.stem] = ign
    (split_dir / "ignore_regions.json").write_text(json.dumps(ignores))

for split, name in SPLITS.items():
    if download_split(name):
        convert(DATA / name)

TRAIN   = DATA / "VisDrone2019-DET-train"
MASKED  = DATA / "train_masked"
TILED   = DATA / "train_tiles"
TILE_SIZES = [640, 960]
OVERLAP    = 0.25
MIN_VIS    = 0.6     # keep a cut box only if >=60% of it is inside the tile
EMPTY_KEEP = 0.1     # keep 10% of tiles that contain no objects
GREY = 114
for d in (MASKED, TILED):
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)

IGN = json.loads((TRAIN / "ignore_regions.json").read_text())

def load_boxes(stem, W, H):
    p = TRAIN / "labels" / f"{stem}.txt"
    rows = [r.split() for r in p.read_text().splitlines() if r.strip()] if p.exists() else []
    out = []
    for c, cx, cy, w, h in rows:
        cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
        out.append([int(c), cx-w/2, cy-h/2, cx+w/2, cy+h/2])
    return np.array(out, dtype=np.float32).reshape(-1, 5)

def starts(L, T, step):
    if L <= T:
        return [0]
    s = list(range(0, L - T + 1, step))
    if s[-1] + T < L:
        s.append(L - T)
    return s

def process(stem):
    rng = random.Random(stem)
    img = cv2.imread(str(TRAIN / "images" / f"{stem}.jpg"))
    H, W = img.shape[:2]
    for x, y, w, h in IGN.get(stem, []):
        img[max(0, y):y+h, max(0, x):x+w] = GREY
    cv2.imwrite(str(MASKED / "images" / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    shutil.copy(TRAIN / "labels" / f"{stem}.txt", MASKED / "labels" / f"{stem}.txt")

    boxes = load_boxes(stem, W, H)
    n = 0
    for T in TILE_SIZES:
        if T >= max(W, H):
            continue
        step = int(T * (1 - OVERLAP))
        for y0 in starts(H, T, step):
            for x0 in starts(W, T, step):
                x1, y1, x2, y2 = x0, y0, min(x0+T, W), min(y0+T, H)
                tile = img[y1:y2, x1:x2].copy()
                tw, th = x2 - x1, y2 - y1
                lines = []
                if len(boxes):
                    bx1 = np.clip(boxes[:, 1], x1, x2); by1 = np.clip(boxes[:, 2], y1, y2)
                    bx2 = np.clip(boxes[:, 3], x1, x2); by2 = np.clip(boxes[:, 4], y1, y2)
                    inter = (bx2 - bx1) * (by2 - by1)
                    area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 4] - boxes[:, 2])
                    vis = np.where(area > 0, inter / area, 0)
                    for i in np.where(inter > 0)[0]:
                        if vis[i] >= MIN_VIS:
                            cx = ((bx1[i] + bx2[i]) / 2 - x1) / tw
                            cy = ((by1[i] + by2[i]) / 2 - y1) / th
                            w_ = (bx2[i] - bx1[i]) / tw
                            h_ = (by2[i] - by1[i]) / th
                            lines.append(f"{int(boxes[i,0])} {cx:.6f} {cy:.6f} {w_:.6f} {h_:.6f}")
                        else:
                            tile[int(by1[i]-y1):int(by2[i]-y1), int(bx1[i]-x1):int(bx2[i]-x1)] = GREY
                if not lines and rng.random() > EMPTY_KEEP:
                    continue
                name = f"{stem}_t{T}_{x0}_{y0}"
                cv2.imwrite(str(TILED / "images" / f"{name}.jpg"), tile, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (TILED / "labels" / f"{name}.txt").write_text("\n".join(lines))
                n += 1
    return n

stems = [p.stem for p in (TRAIN / "images").glob("*.jpg")]
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    counts = list(tqdm(ex.map(process, stems, chunksize=16), total=len(stems), desc="masking + tiling"))
print(f"D1: {len(stems)} masked images | D2 tiles: {sum(counts)}   (K1 had 6471 | 63745)")

VAL = "VisDrone2019-DET-val/images"
TEST = "VisDrone2019-DET-test-dev/images"
names_block = "names:\n" + "\n".join(f"  {i}: {n}" for i, n in enumerate(NAMES))
(DATA / "d2_tiles.yaml").write_text(
    f"path: {DATA}\ntrain:\n  - train_masked/images\n  - train_tiles/images\nval: {VAL}\ntest: {TEST}\n{names_block}")
print("written", DATA / "d2_tiles.yaml")

VisDrone2019-DET-train: 100%|██████████| 6471/6471 [00:02<00:00, 2220.71it/s]


VisDrone2019-DET-val: 100%|██████████| 548/548 [00:00<00:00, 1980.95it/s]


masking + tiling: 100%|██████████| 6471/6471 [03:07<00:00, 34.46it/s]

D1: 6471 masked images | D2 tiles: 63745   (K1 had 6471 | 63745)
written /tmp/VisDrone/d2_tiles.yaml


In [4]:
# --- Config (E2 run) -------------------------------------------------------
from pathlib import Path
WORK = Path("/kaggle/working")
RUNS = WORK / "runs_yoloe"

VAL_IMG_DIR  = "/tmp/VisDrone_raw/VisDrone2019-DET-val/images"
VAL_ANN_DIR  = "/tmp/VisDrone_raw/VisDrone2019-DET-val/annotations"
TEST_IMG_DIR = "/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/images"
TEST_ANN_DIR = "/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/annotations"
DATA_YAML = "/tmp/VisDrone/d2_tiles.yaml"

MODEL_YAML = "yoloe-26m.yaml"
PRETRAINED = "yoloe-26m-seg.pt"
DEVICE = "0,1"
IMGSZ_TRAIN = 1024
EVAL_SIZES = [1536]
SAHI_SLICE = 512

# K1's settings, copied from its checkpoint (so this notebook needs no inputs attached)
K1_HP = {'batch': 16, 'workers': 4, 'mosaic': 1.0, 'close_mosaic': 3, 'mixup': 0.1, 'copy_paste': 0.0,
         'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0,
         'fliplr': 0.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'erasing': 0.4, 'multi_scale': 0.0,
         'cos_lr': True, 'lrf': 0.01}
E2_EPOCHS, E2_HOURS = 12, 10.5     # same budget as K1; drop to 10.0 if tile building + eval run long

VD_CLASSES = ["pedestrian", "people", "bicycle", "car", "van",
              "truck", "tricycle", "awning-tricycle", "bus", "motor"]
RESULTS = []

In [5]:
# --- Helpers: GT loading, quick scorer, full-image + SAHI prediction, latency ---
import contextlib, io, shutil, time, json
import numpy as np, torch, torchvision
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO, YOLOE

CUDA = torch.cuda.is_available()
DEV0 = 0 if CUDA else "cpu"
HALF = CUDA

def load_visdrone_gt(img_dir, ann_dir):
    # VisDrone txt: x,y,w,h,score,category,truncation,occlusion
    # cat 0 = ignored region, 11 = others (not evaluated), 1..10 = classes
    images, anns, ignores, aid = [], [], {}, 1
    for i, p in enumerate(sorted(Path(img_dir).glob("*.jpg"))):
        w, h = Image.open(p).size
        images.append(dict(id=i, file_name=p.name, width=w, height=h))
        ig, objs = [], []
        ann = Path(ann_dir) / f"{p.stem}.txt"
        if ann.exists():
            for line in ann.read_text().splitlines():
                v = [int(float(x)) for x in line.strip().rstrip(",").split(",")[:6]] if line.strip() else None
                if not v: continue
                x, y, bw, bh, _, cat = v
                if bw <= 0 or bh <= 0: continue
                if cat == 0: ig.append([x, y, x + bw, y + bh])
                elif 1 <= cat <= 10: objs.append([x, y, bw, bh, cat])
        ig = np.array(ig, dtype=np.float32).reshape(-1, 4)
        ignores[i] = ig
        for x, y, bw, bh, cat in objs:
            if len(ig) and _ign_frac(np.array([[x, y, x + bw, y + bh]], np.float32), ig)[0] > 0.5:
                continue   # GT inside ignore region is dropped, as in the official toolkit
            anns.append(dict(id=aid, image_id=i, category_id=cat, bbox=[x, y, bw, bh], area=bw * bh, iscrowd=0)); aid += 1
    gt = dict(images=images, annotations=anns,
              categories=[dict(id=k + 1, name=n) for k, n in enumerate(VD_CLASSES)])
    return gt, ignores

def _ign_frac(boxes, ig):
    # fraction of each box's area covered by any single ignore region (max over regions)
    if len(ig) == 0 or len(boxes) == 0: return np.zeros(len(boxes))
    x1 = np.maximum(boxes[:, None, 0], ig[None, :, 0]); y1 = np.maximum(boxes[:, None, 1], ig[None, :, 1])
    x2 = np.minimum(boxes[:, None, 2], ig[None, :, 2]); y2 = np.minimum(boxes[:, None, 3], ig[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = np.clip((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]), 1e-6, None)
    return (inter / area[:, None]).max(1)

def quick_score(gt, ignores, dets, label=None, **meta):
    # dets: list of dict(image_id, category_id 1..10, bbox xywh, score). Drops dets >50% inside ignore regions.
    keep = []
    by_img = {}
    for d in dets: by_img.setdefault(d["image_id"], []).append(d)
    for iid, ds in by_img.items():
        b = np.array([[d["bbox"][0], d["bbox"][1], d["bbox"][0] + d["bbox"][2], d["bbox"][1] + d["bbox"][3]] for d in ds], np.float32)
        frac = _ign_frac(b, ignores.get(iid, np.zeros((0, 4))))
        keep += [d for d, f in zip(ds, frac) if f <= 0.5]
    out = dict(run=label, **meta, AP=0.0, AP50=0.0, AP75=0.0, APs=0.0, APm=0.0, APl=0.0)
    if keep:
        with contextlib.redirect_stdout(io.StringIO()):
            G = COCO(); G.dataset = gt; G.createIndex()
            E = COCOeval(G, G.loadRes(keep), "bbox")
            E.params.maxDets = [1, 100, 500]   # VisDrone evaluates up to 500 dets/image
            E.evaluate(); E.accumulate(); E.summarize()
        s = E.stats
        out.update(AP=s[0], AP50=s[1], AP75=s[2], APs=s[3], APm=s[4], APl=s[5])
        p = E.eval["precision"][0, :, :, 0, 2]   # IoU .5, all area, maxDets 500
        for k, n in enumerate(VD_CLASSES):
            pk = p[:, k]; out[f"AP50_{n}"] = float(pk[pk > -1].mean()) if (pk > -1).any() else 0.0
    if label: RESULTS.append(out)
    print(f"{label}: AP50={out['AP50']:.3f}  AP={out['AP']:.3f}  APs={out['APs']:.3f}  ({len(keep)} dets kept)")
    return out

def _merge_same_class(dets, iou=0.6):
    # used when several prompts map to one class: suppress duplicates across synonyms
    if not dets: return dets
    b = torch.tensor([[d["bbox"][0], d["bbox"][1], d["bbox"][0] + d["bbox"][2], d["bbox"][1] + d["bbox"][3]] for d in dets])
    s = torch.tensor([d["score"] for d in dets]); c = torch.tensor([d["category_id"] for d in dets])
    k = torchvision.ops.batched_nms(b.float(), s.float(), c, iou)
    return [dets[i] for i in k.tolist()]

def predict_full(model, gt, img_dir, imgsz, cls_map=None, conf=0.001, max_det=1000, save_txt_dir=None):
    # cls_map: model class index -> VisDrone category id (1..10). None = identity (+1).
    dets, merge = [], cls_map is not None and len(set(cls_map.values())) < len(cls_map)
    if save_txt_dir: Path(save_txt_dir).mkdir(parents=True, exist_ok=True)
    for im in gt["images"]:
        r = model.predict(str(Path(img_dir) / im["file_name"]), imgsz=imgsz, conf=conf, max_det=max_det,
                          half=HALF, device=DEV0, verbose=False)[0]
        xyxy = r.boxes.xyxy.cpu().numpy(); sc = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy().astype(int)
        ds = [dict(image_id=im["id"], category_id=(cls_map[c] if cls_map else c + 1),
                   bbox=[float(a), float(b), float(c2 - a), float(d - b)], score=float(s))
              for (a, b, c2, d), s, c in zip(xyxy, sc, cl)]
        if merge: ds = _merge_same_class(ds)
        dets += ds
        if save_txt_dir:   # VisDrone result format, for the eval notebook
            with open(Path(save_txt_dir) / f"{Path(im['file_name']).stem}.txt", "w") as f:
                for d in ds:
                    x, y, w, h = d["bbox"]; f.write(f"{x:.1f},{y:.1f},{w:.1f},{h:.1f},{d['score']:.4f},{d['category_id']},-1,-1\n")
    return dets

def predict_sahi(model, gt, img_dir, cls_map=None, slice_size=SAHI_SLICE, overlap=0.2, conf=0.001):
    # Same defaults as SAHI's get_sliced_prediction (full-image pass on, GREEDYNMM / IOS 0.5).
    # Match these to your eval notebook's SAHI settings if they differ.
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    sm = AutoDetectionModel.from_pretrained(model_type="ultralytics", model=model, confidence_threshold=conf,
                                            device="cuda:0" if CUDA else "cpu", image_size=slice_size)
    dets, merge = [], cls_map is not None and len(set(cls_map.values())) < len(cls_map)
    for im in gt["images"]:
        res = get_sliced_prediction(str(Path(img_dir) / im["file_name"]), sm, slice_height=slice_size, slice_width=slice_size,
                                    overlap_height_ratio=overlap, overlap_width_ratio=overlap, verbose=0)
        ds = []
        for op in res.object_prediction_list:
            c = op.category.id; b = op.bbox
            ds.append(dict(image_id=im["id"], category_id=(cls_map[c] if cls_map else c + 1),
                           bbox=[b.minx, b.miny, b.maxx - b.minx, b.maxy - b.miny], score=float(op.score.value)))
        if merge: ds = _merge_same_class(ds)
        dets += ds
    return dets

def latency_ms(model, img_dir, imgsz, n=100):
    paths = sorted(Path(img_dir).glob("*.jpg"))[:n]
    for p in paths[:5]: model.predict(str(p), imgsz=imgsz, half=HALF, device=DEV0, verbose=False)   # warm-up
    if CUDA: torch.cuda.synchronize()
    t = time.perf_counter()
    for p in paths: model.predict(str(p), imgsz=imgsz, half=HALF, device=DEV0, verbose=False)
    if CUDA: torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t) / len(paths)

def promptable(weights, prompts):
    m = YOLOE(weights); m.set_classes(list(prompts)); return m

VAL_GT, VAL_IGN = load_visdrone_gt(VAL_IMG_DIR, VAL_ANN_DIR)
print(len(VAL_GT["images"]), "val images,", len(VAL_GT["annotations"]), "GT boxes")

548 val images, 38610 GT boxes


In [6]:
import os
# SAHI (used in the E0 / eval cells) sets CUDA_VISIBLE_DEVICES="0" for the whole session, which hides
# the second T4 from the DDP worker processes. Undo it before multi-GPU training.
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
assert Path(DATA_YAML).exists(), "Build cell must run first"
from ultralytics.models.yolo.yoloe import YOLOEPETrainer
m = YOLOE(MODEL_YAML); m.load(PRETRAINED)
m.train(data=DATA_YAML, trainer=YOLOEPETrainer,
        epochs=E2_EPOCHS, time=E2_HOURS, imgsz=IMGSZ_TRAIN, device=DEVICE,
        optimizer="AdamW", lr0=1e-3, warmup_bias_lr=0.0, weight_decay=0.025, momentum=0.9,
        project=str(RUNS), name="e2_full", exist_ok=True, save_period=1, **K1_HP)
E2_W = WORK / "yoloe26m_e2_full.pt"
shutil.copy(RUNS / "e2_full/weights/best.pt", E2_W)

Transferred 882/882 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/VisDrone/d2_tiles.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024,

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/14      13.3G      1.667      1.381   0.003125         51       1024: 100% ━━━━━━━━━━━━ 4389/4389 1.6it/s 45:57
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.4it/s 7.4s
                   all        548      38759      0.591      0.465      0.492      0.297

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/14      13.8G      1.643      1.328   0.003059         52       1024: 100% ━━━━━━━━━━━━ 4389/4389 1.6it/s 45:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.3it/s 7.7s
                   all        548      38759      0.624      0.483      0.522      0.321

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/14      13.8G      1.614      1.257   0.002944        103       1024: 100% ━━━━━

PosixPath('/kaggle/working/yoloe26m_e2_full.pt')

In [7]:
# E2 eval. If E2 beats K1 here, run test-dev ONCE for the chosen config (val is for picking, test-dev for reporting).
E2_W = WORK / "yoloe26m_e2_full.pt"
m = YOLOE(E2_W)
assert list(m.names.values()) == VD_CLASSES, m.names
for sz in EVAL_SIZES:
    quick_score(VAL_GT, VAL_IGN, predict_full(m, VAL_GT, VAL_IMG_DIR, sz, save_txt_dir=WORK / f"preds/e2_val_full{sz}"),
                label="E2 full FT", mode=f"full@{sz}", ms=latency_ms(m, VAL_IMG_DIR, sz))
quick_score(VAL_GT, VAL_IGN, predict_sahi(m, VAL_GT, VAL_IMG_DIR), label="E2 full FT", mode=f"SAHI-{SAHI_SLICE}")

RUN_TEST_DEV = False
if RUN_TEST_DEV:
    TEST_GT, TEST_IGN = load_visdrone_gt(TEST_IMG_DIR, TEST_ANN_DIR)
    quick_score(TEST_GT, TEST_IGN, predict_sahi(m, TEST_GT, TEST_IMG_DIR), label="E2 full FT [test-dev]", mode=f"SAHI-{SAHI_SLICE}")
del m; torch.cuda.empty_cache()

import pandas as pd
pd.DataFrame(RESULTS).to_csv(WORK / "e2_results.csv", index=False)   # RESULTS is lost when the session ends
print("saved", WORK / "e2_results.csv")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

2026-10-13 00:37:21,847 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:22,606 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:22,954 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:23,319 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:23,688 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:24,046 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-13 00:37:24,396 - sahi - WARNING - Switching postprocess type/metric

E2 full FT: AP50=0.566  AP=0.344  APs=0.264  (291976 dets kept)
saved /kaggle/working/e2_results.csv
